# Decision Tree Classification with the Iris Dataset
# Data and Classification task
The Iris dataset contrains 150 flower measurements from versicolor, virginica and setosa. 

Each sample has four numeric features: 
- Sepal width
- Sepal length
- Petal width
- Petal length

The label is based on the flower species, and the classification task is to predict the species from the four numeric features. 

A decision tree is a resonable way to look at this since it can learn simple threshold rules, and work directly with numeric data while not requrieing feature scaling. 

In [3]:
import sys
print(sys.executable)

/Users/jasvz/miniforge3/bin/python


In [ ]:
# imports data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

print(X.head())
print(iris.target_names)
print(X.shape)

In [ ]:
# train, validation, and test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("Testing samples:", len(X_test))

The data is split into 20% validation, 60% training data, and 20% testing data. 

The training set is being used to fit models. While the validation set is used to tune hyperparameters. With the testing set being kept seperate until the final evaluation. 

In [ ]:
# baseline decision tree
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_model.fit(X_train, y_train)

validation_predictions = baseline_model.predict(X_validation)
baseline_accuracy = accuracy_score(
    y_validation, 
    validation_predictions
)

print("Baseline validation accuracy: ", baseline_accuracy)

In [ ]:
# cell tune "max_depth"

max_depth_values = range(1, 11)
max_depth_scores = []

for depth in max_depth_values:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)
    score = accuracy_score(y_validation, predictions)

    max_depth_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(
    list(max_depth_values),
    max_depth_scores,
    marker="o"
)
plt.xlabel("Maximum Tree Depth")
plt.ylabel("Validation Accuracy")
plt.title("Maximum Depth vs. Validation Accuracy")
plt.xticks(list(max_depth_values))
plt.grid(True)
plt.show()


In [ ]:
# tune "min_samples_leaf" 

leaf_values = [1, 2, 3, 4, 5, 6, 8, 10]
leaf_scores = []

for leaf_size in leaf_values:
    model = DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=leaf_size,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)
    score = accuracy_score(y_validation, predictions)

    leaf_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(
    leaf_values,
    leaf_scores,
    marker="o"
)
plt.xlabel("Minimum Samples per Leaf")
plt.ylabel("Validation Accuracy")
plt.title("Minimum Samples per Leaf vs. Validation Accuracy")
plt.grid(True)
plt.show()

In [ ]:
# selecting optimal hyperparameters

best_depth = list(max_depth_values)[
    np.argmax(max_depth_scores)
]

best_leaf_size = leaf_values[
    np.argmax(leaf_scores)
]

print("Best max_depth:", best_depth)
print("Best min_samples_leaf:", best_leaf_size)

The best hyperparameter values are selected based on the highest validation accuracy.

The `max_depth` parameter controls how deep the tree can grow. Smaller values produce simpler models, while larger values allow more complex rules.

The `min_samples_leaf` parameter controls the minimum number of observations allowed in a leaf. Increasing this value can reduce overfitting by preventing very small leaves.

In [ ]:
# train final model 

final_model = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_leaf=best_leaf_size,
    random_state=42
)

final_model.fit(X_train_val, y_train_val)

test_predictions = final_model.predict(X_test)

print(classification_report(
    y_test,
    test_predictions,
    target_names=iris.target_names
))

In [ ]:
# confustion matrix 

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=iris.target_names,
    cmap="Blues"
)

plt.title("Test Set Confusion Matrix")
plt.show()

In [ ]:
# to visualize the decision tree 

plt.figure(figsize=(18, 10))

plot_tree(
    final_model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Final Decision Tree")
plt.show()

# Model interpretation 

The decision tree primarily uses petal measurements to distinguish the species. This is reasonable because petal length and petal width are strongly different among the three Iris species. 

The first splits in the tree represent the most useful features for classifiction. With each internal node containing a threshold rule. E.g. the tree may ask whether petal width is less that a specifiyed value. 

The leaves of the tree show the predicited class. The colors indicating the dominant class in that leaf, while darker colors indicate the leaf continaing mostly one species. 

The confusion matrix and classification report show how well the final model performs on previously unseen testing data. 